# Actuarial Pricing and Reserving Notebook (Expanded)

This notebook integrates pricing and reserving into a single actuarial decision workflow.

What is included:
- synthetic policy portfolio with claim frequency and severity,
- frequency and severity model benchmarking,
- pure premium and indicated premium calculation,
- portfolio segmentation and rate adequacy diagnostics,
- paid-loss triangle reserving with bootstrap uncertainty,
- combined pricing-reserving capital view.

## 0) Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(42)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

## 1) Simulate an insurance portfolio

In [ ]:
n = 22000

portfolio = pd.DataFrame(
    {
        "policy_id": np.arange(1, n + 1),
        "driver_age": np.random.randint(18, 86, size=n),
        "vehicle_age": np.random.randint(0, 21, size=n),
        "vehicle_power": np.random.normal(120, 35, size=n).clip(50, 280),
        "annual_mileage_k": np.random.normal(14, 5, size=n).clip(3, 45),
        "tenure_years": np.random.exponential(4, size=n).clip(0.1, 20),
        "region": np.random.choice(["North", "South", "East", "West"], p=[0.23, 0.31, 0.24, 0.22], size=n),
        "segment": np.random.choice(["standard", "preferred", "high_risk"], p=[0.62, 0.22, 0.16], size=n),
        "distribution": np.random.choice(["agent", "online", "broker"], p=[0.50, 0.30, 0.20], size=n),
        "exposure": np.random.uniform(0.5, 1.0, size=n),
    }
)

# Frequency latent process
freq_linear = (
    -2.25
    + 0.012 * np.maximum(0, 25 - portfolio["driver_age"].to_numpy())
    + 0.010 * portfolio["vehicle_age"].to_numpy()
    + 0.018 * (portfolio["annual_mileage_k"].to_numpy() - 12)
    + 0.11 * (portfolio["segment"].to_numpy() == "high_risk")
    - 0.06 * (portfolio["segment"].to_numpy() == "preferred")
    + 0.08 * (portfolio["region"].to_numpy() == "South")
)

lambda_claims = np.exp(freq_linear) * portfolio["exposure"].to_numpy()
portfolio["claim_count"] = np.random.poisson(lambda_claims)

# Severity latent process (for claims > 0)
sev_linear = (
    8.05
    + 0.0025 * portfolio["vehicle_power"].to_numpy()
    + 0.028 * portfolio["vehicle_age"].to_numpy()
    + 0.12 * (portfolio["segment"].to_numpy() == "high_risk")
    + 0.06 * (portfolio["region"].to_numpy() == "West")
)

avg_sev = np.exp(np.random.normal(loc=sev_linear, scale=0.42, size=n))
portfolio["avg_claim_severity"] = avg_sev
portfolio["total_claim_amount"] = portfolio["claim_count"] * portfolio["avg_claim_severity"]
portfolio["has_claim"] = (portfolio["claim_count"] > 0).astype(int)

portfolio.head()

In [ ]:
portfolio_summary = pd.Series(
    {
        "policies": len(portfolio),
        "total_exposure": portfolio["exposure"].sum(),
        "claim_frequency_per_policy": portfolio["claim_count"].mean(),
        "claiming_policy_rate": portfolio["has_claim"].mean(),
        "avg_severity_on_policies": portfolio.loc[portfolio["claim_count"] > 0, "avg_claim_severity"].mean(),
        "loss_ratio_base_premium_2000": portfolio["total_claim_amount"].sum() / (2000 * portfolio["exposure"].sum()),
    }
)
portfolio_summary

## 2) Exploratory diagnostics

In [ ]:
risk_by_segment = (
    portfolio.groupby("segment", as_index=False)
    .agg(
        exposure=("exposure", "sum"),
        frequency=("claim_count", "mean"),
        severity=("avg_claim_severity", "mean"),
        pure_premium=("total_claim_amount", "sum"),
    )
)
risk_by_segment["pure_premium_per_exposure"] = risk_by_segment["pure_premium"] / risk_by_segment["exposure"]
risk_by_segment

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

sns.barplot(data=risk_by_segment, x="segment", y="frequency", ax=ax[0], palette="Blues")
ax[0].set_title("Claim Frequency by Segment")

sns.barplot(data=risk_by_segment, x="segment", y="pure_premium_per_exposure", ax=ax[1], palette="Reds")
ax[1].set_title("Pure Premium per Exposure")

plt.tight_layout()
plt.show()

## 3) Train/test split and feature setup

In [ ]:
feature_cols = [
    "driver_age",
    "vehicle_age",
    "vehicle_power",
    "annual_mileage_k",
    "tenure_years",
    "region",
    "segment",
    "distribution",
    "exposure",
]

X = portfolio[feature_cols]
y_freq = portfolio["claim_count"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_freq, test_size=0.30, random_state=42
)

num_cols = ["driver_age", "vehicle_age", "vehicle_power", "annual_mileage_k", "tenure_years", "exposure"]
cat_cols = ["region", "segment", "distribution"]

prep = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

## 4) Frequency model benchmarking

In [ ]:
freq_poisson = Pipeline(
    [
        ("prep", prep),
        ("model", PoissonRegressor(alpha=0.2, max_iter=400)),
    ]
)

freq_hgb = Pipeline(
    [
        ("prep", prep),
        (
            "model",
            HistGradientBoostingRegressor(
                loss="poisson",
                learning_rate=0.07,
                max_depth=5,
                min_samples_leaf=60,
                max_iter=260,
                random_state=42,
            ),
        ),
    ]
)

freq_poisson.fit(X_train, y_train)
freq_hgb.fit(X_train, y_train)

pred_freq_poisson = np.clip(freq_poisson.predict(X_test), 0, None)
pred_freq_hgb = np.clip(freq_hgb.predict(X_test), 0, None)

freq_metrics = pd.DataFrame(
    [
        {
            "model": "PoissonRegressor",
            "mae": mean_absolute_error(y_test, pred_freq_poisson),
            "rmse": np.sqrt(mean_squared_error(y_test, pred_freq_poisson)),
        },
        {
            "model": "HistGB-Poisson",
            "mae": mean_absolute_error(y_test, pred_freq_hgb),
            "rmse": np.sqrt(mean_squared_error(y_test, pred_freq_hgb)),
        },
    ]
).sort_values("rmse")
freq_metrics

## 5) Severity model on claiming policies

In [ ]:
sev_df = portfolio[portfolio["claim_count"] > 0].copy()
X_sev = sev_df[feature_cols]
y_sev = np.log1p(sev_df["avg_claim_severity"])  # stabilize tail

X_sev_train, X_sev_test, y_sev_train, y_sev_test = train_test_split(
    X_sev, y_sev, test_size=0.30, random_state=42
)

sev_model = Pipeline(
    [
        ("prep", prep),
        (
            "model",
            HistGradientBoostingRegressor(
                learning_rate=0.06,
                max_depth=4,
                min_samples_leaf=45,
                max_iter=280,
                random_state=42,
            ),
        ),
    ]
)

sev_model.fit(X_sev_train, y_sev_train)
sev_pred_log = sev_model.predict(X_sev_test)
sev_pred = np.expm1(sev_pred_log)
sev_true = np.expm1(y_sev_test)

sev_metrics = pd.Series(
    {
        "severity_mae": mean_absolute_error(sev_true, sev_pred),
        "severity_rmse": np.sqrt(mean_squared_error(sev_true, sev_pred)),
    }
)
sev_metrics

## 6) Pure premium modeling and adequacy diagnostics

In [ ]:
best_freq_name = freq_metrics.iloc[0]["model"]
freq_model = freq_poisson if best_freq_name == "PoissonRegressor" else freq_hgb

portfolio_scored = portfolio.copy()
portfolio_scored["freq_hat"] = np.clip(freq_model.predict(portfolio_scored[feature_cols]), 0, None)
portfolio_scored["sev_hat"] = np.expm1(sev_model.predict(portfolio_scored[feature_cols]))
portfolio_scored["pure_premium_hat"] = portfolio_scored["freq_hat"] * portfolio_scored["sev_hat"]
portfolio_scored["pure_premium_actual"] = portfolio_scored["total_claim_amount"] / portfolio_scored["exposure"]

pricing_diag = pd.Series(
    {
        "selected_frequency_model": best_freq_name,
        "avg_pred_pure_premium": portfolio_scored["pure_premium_hat"].mean(),
        "avg_actual_pure_premium": portfolio_scored["pure_premium_actual"].mean(),
        "premium_gap_pct": (
            portfolio_scored["pure_premium_hat"].mean() / portfolio_scored["pure_premium_actual"].mean() - 1
        )
        * 100,
    }
)
pricing_diag

In [ ]:
# Convert pure premium into indicated written premium
expense_load = 0.22
profit_load = 0.06
cat_load_by_region = {"North": 0.015, "South": 0.03, "East": 0.02, "West": 0.025}

portfolio_scored["cat_load"] = portfolio_scored["region"].map(cat_load_by_region)
portfolio_scored["technical_premium"] = portfolio_scored["pure_premium_hat"] / (1 - expense_load - profit_load - portfolio_scored["cat_load"])

portfolio_scored[["pure_premium_hat", "technical_premium"]].describe().T

In [ ]:
rate_adequacy = (
    portfolio_scored.groupby(["segment", "region"], as_index=False)
    .agg(
        exposure=("exposure", "sum"),
        avg_indicated_premium=("technical_premium", "mean"),
        avg_actual_loss=("pure_premium_actual", "mean"),
    )
)
rate_adequacy["rate_adequacy_ratio"] = rate_adequacy["avg_indicated_premium"] / rate_adequacy["avg_actual_loss"]
rate_adequacy.sort_values("rate_adequacy_ratio")

In [ ]:
plt.figure(figsize=(10, 4.8))
plot_df = rate_adequacy.sort_values("rate_adequacy_ratio").copy()
plot_df["cell"] = plot_df["segment"] + " | " + plot_df["region"]

sns.barplot(data=plot_df, x="rate_adequacy_ratio", y="cell", palette="viridis")
plt.axvline(1.0, linestyle="--", color="black")
plt.title("Rate Adequacy Ratio by Segment-Region Cell")
plt.xlabel("Indicated premium / Actual pure premium")
plt.ylabel("Cell")
plt.tight_layout()
plt.show()

## 7) Build paid-loss triangle for reserving

In [ ]:
accident_years = np.arange(2015, 2027)
dev_periods = np.arange(1, 13)

ult = np.random.lognormal(mean=10.7, sigma=0.23, size=len(accident_years)) * np.linspace(0.95, 1.08, len(accident_years))
cdf_pattern = np.array([0.17, 0.31, 0.45, 0.58, 0.68, 0.77, 0.84, 0.89, 0.93, 0.965, 0.985, 1.0])

triangle = pd.DataFrame(index=accident_years, columns=dev_periods, dtype=float)
for i, ay in enumerate(accident_years):
    max_dev = len(accident_years) - i
    for d in dev_periods[:max_dev]:
        noise = np.random.normal(1.0, 0.024)
        triangle.loc[ay, d] = ult[i] * cdf_pattern[d - 1] * noise
triangle = triangle.cummax(axis=1)

triangle.head()

## 8) Deterministic chain-ladder reserve

In [ ]:
# link ratios
factors = {}
for d in range(1, triangle.shape[1]):
    num = den = 0.0
    for ay in accident_years:
        if pd.notna(triangle.loc[ay, d]) and pd.notna(triangle.loc[ay, d + 1]):
            num += triangle.loc[ay, d + 1]
            den += triangle.loc[ay, d]
    factors[d] = num / den

f = pd.Series(factors)
cdf = {triangle.shape[1]: 1.0}
prod = 1.0
for d in range(triangle.shape[1] - 1, 0, -1):
    prod *= f[d]
    cdf[d] = prod
cdf = pd.Series(cdf).sort_index()

rows = []
for i, ay in enumerate(accident_years):
    latest_dev = len(accident_years) - i
    latest_paid = triangle.loc[ay, latest_dev]
    ultimate = latest_paid * cdf[latest_dev]
    ibnr = ultimate - latest_paid
    rows.append({
        "accident_year": ay,
        "latest_dev": latest_dev,
        "latest_paid": latest_paid,
        "ultimate": ultimate,
        "ibnr": ibnr,
    })

cl = pd.DataFrame(rows)
cl

In [ ]:
total_ibnr = cl["ibnr"].sum()

plt.figure(figsize=(8.8, 4.5))
plt.bar(cl["accident_year"].astype(str), cl["ibnr"], color="#4C72B0")
plt.title("IBNR by Accident Year (Deterministic CL)")
plt.xlabel("Accident year")
plt.ylabel("IBNR")
plt.tight_layout()
plt.show()

print(f"Deterministic total IBNR: {total_ibnr:,.0f}")

## 9) Bootstrap reserve uncertainty

In [ ]:
ratio_pool = {}
for d in range(1, triangle.shape[1]):
    ratio_pool[d] = np.array(
        [
            triangle.loc[ay, d + 1] / triangle.loc[ay, d]
            for ay in accident_years
            if pd.notna(triangle.loc[ay, d]) and pd.notna(triangle.loc[ay, d + 1])
        ]
    )

n_boot = 1200
boot_ibnr = []

for _ in range(n_boot):
    bf = {}
    for d in range(1, triangle.shape[1]):
        bf[d] = np.mean(np.random.choice(ratio_pool[d], size=max(20, len(ratio_pool[d])), replace=True)) * np.random.normal(1.0, 0.01)

    bcdf = {triangle.shape[1]: 1.0}
    prod = 1.0
    for d in range(triangle.shape[1] - 1, 0, -1):
        prod *= bf[d]
        bcdf[d] = prod

    total = 0.0
    for i, ay in enumerate(accident_years):
        latest_dev = len(accident_years) - i
        latest_paid = triangle.loc[ay, latest_dev]
        total += latest_paid * bcdf[latest_dev] - latest_paid

    boot_ibnr.append(total)

boot_ibnr = np.array(boot_ibnr)

reserve_dist = pd.Series(
    {
        "mean": np.mean(boot_ibnr),
        "std": np.std(boot_ibnr, ddof=1),
        "p75": np.quantile(boot_ibnr, 0.75),
        "p90": np.quantile(boot_ibnr, 0.90),
        "p95": np.quantile(boot_ibnr, 0.95),
    }
)
reserve_dist

In [ ]:
plt.figure(figsize=(8.5, 4.5))
plt.hist(boot_ibnr, bins=38, color="#8172B2", alpha=0.85)
plt.axvline(total_ibnr, linestyle="--", color="black", label="Deterministic")
plt.axvline(np.quantile(boot_ibnr, 0.95), linestyle="--", color="#C44E52", label="P95")
plt.title("Bootstrap Distribution of Total IBNR")
plt.xlabel("Total IBNR")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## 10) Combined pricing + reserving governance view

In [ ]:
written_premium = (portfolio_scored["technical_premium"] * portfolio_scored["exposure"]).sum()
expected_loss_cost = (portfolio_scored["pure_premium_hat"] * portfolio_scored["exposure"]).sum()
reserve_p75 = np.quantile(boot_ibnr, 0.75)
reserve_p95 = np.quantile(boot_ibnr, 0.95)

capital_view = pd.Series(
    {
        "written_premium": written_premium,
        "expected_loss_cost_pricing": expected_loss_cost,
        "pricing_margin": written_premium - expected_loss_cost,
        "deterministic_ibnr": total_ibnr,
        "ibnr_p75": reserve_p75,
        "ibnr_p95": reserve_p95,
        "combined_margin_after_p75_reserve": written_premium - expected_loss_cost - reserve_p75,
        "combined_margin_after_p95_reserve": written_premium - expected_loss_cost - reserve_p95,
    }
)
capital_view

## 11) Final summary

- Pricing and reserving are coupled; premium adequacy should be judged jointly with reserve uncertainty.
- Frequency/severity decomposition improves pricing transparency and supports rate filing narrative.
- Bootstrap reserve percentiles provide actionable risk-margin options for governance committees.
- The combined capital view helps translate technical actuarial outputs into management decisions.